In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import sklearn

In [20]:
import pandas as pd
import numpy as np
import seaborn as sns

In [21]:
data = pd.read_csv('../data/student_data.csv')

In [22]:
data.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [23]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      395 non-null    str  
 1   sex         395 non-null    str  
 2   age         395 non-null    int64
 3   address     395 non-null    str  
 4   famsize     395 non-null    str  
 5   Pstatus     395 non-null    str  
 6   Medu        395 non-null    int64
 7   Fedu        395 non-null    int64
 8   Mjob        395 non-null    str  
 9   Fjob        395 non-null    str  
 10  reason      395 non-null    str  
 11  guardian    395 non-null    str  
 12  traveltime  395 non-null    int64
 13  studytime   395 non-null    int64
 14  failures    395 non-null    int64
 15  schoolsup   395 non-null    str  
 16  famsup      395 non-null    str  
 17  paid        395 non-null    str  
 18  activities  395 non-null    str  
 19  nursery     395 non-null    str  
 20  higher      395 non-null    str  
 21  inte

In [24]:
data.isnull().sum().sum()

np.int64(0)

In [25]:
data_encoded = pd.get_dummies(data)

In [26]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 59 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   age                395 non-null    int64
 1   Medu               395 non-null    int64
 2   Fedu               395 non-null    int64
 3   traveltime         395 non-null    int64
 4   studytime          395 non-null    int64
 5   failures           395 non-null    int64
 6   famrel             395 non-null    int64
 7   freetime           395 non-null    int64
 8   goout              395 non-null    int64
 9   Dalc               395 non-null    int64
 10  Walc               395 non-null    int64
 11  health             395 non-null    int64
 12  absences           395 non-null    int64
 13  G1                 395 non-null    int64
 14  G2                 395 non-null    int64
 15  G3                 395 non-null    int64
 16  school_GP          395 non-null    bool 
 17  school_MS          395 non-

In [27]:
# still some non float value (bool type, pytorch cannot directly handle bool)
data_encoded = data_encoded.astype(float)

In [28]:
class FCNN(nn.Module):
    def __init__(self,input_size,hidden_size_1,hidden_size_2, output_size):
        super(FCNN,self).__init__()

        self.fc1 = nn.Linear(input_size, hidden_size_1)

        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        
        self.fc3 = nn.Linear(hidden_size_2,output_size)

    def forward(self,X):
        X = self.fc1(X)
        X = F.relu(X)
        X = self.fc2(X)
        X = F.relu(X)
        X = self.fc3(X)
        return X

### Model training

In [29]:
# covert data from datafram into tensor -- dataframe not convert directly first convert numpy and then convet tensor
# Features
X = data_encoded.drop('G3', axis=1).values
X_tensor = torch.tensor(X, dtype=torch.float32)

# Target
y = data_encoded['G3'].values
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1,1)

In [ ]:
# spliting data set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

In [ ]:
# Model
input_size = X_tensor.shape[1]
model = FCNN(input_size, 64, 32, 1)
print(f"model structure": model)
print(input_size)

FCNN(
  (fc1): Linear(in_features=58, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)
58


In [32]:
# Loss & Optimizer
criterion = nn.MSELoss()   # regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [33]:
# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    
    output = model(X_train)
    loss = criterion(output, y_train)
    
    loss.backward()
    optimizer.step()


### Model Testing

In [35]:
model.eval()

with torch.no_grad():
    predictions = model(X_test)

### Evaluation (RMSE and R²)

In [41]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_test_np = y_test.detach().numpy() # Sklearn works with NumPy arrays
pred_np = predictions.detach().numpy()

mse = mean_squared_error(y_test_np, pred_np)
rmse = np.sqrt(mse)

r2 = r2_score(y_test_np, pred_np)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 2.676746678738018
R2: 0.6505752205848694


### Check Predictions

In [37]:
print(predictions[:5])
print(y_test[:5])

tensor([[ 7.7053],
        [11.2132],
        [ 6.3343],
        [ 9.4026],
        [10.1180]])
tensor([[10.],
        [12.],
        [ 5.],
        [10.],
        [ 9.]])
